In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer

print("✅ Bibliothèques NLP importées")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


✅ Bibliothèques NLP importées


In [3]:
df = pd.read_csv('../data/processed/dataset_clean.csv')
df_nlp = df[df['Categorie'].notna()].copy()

print(f"Shape : {df_nlp.shape}")
print(f"\nExemples de textes :")
for i in range(3):
    print(f"\n→ Categorie : {df_nlp['Categorie'].iloc[i]}")
    print(f"  Texte : {df_nlp['Rapport_Collecte'].iloc[i][:100]}...")

Shape : (6791, 11)

Exemples de textes :

→ Categorie : Papier
  Texte : Lot de papier récupéré dans un site non renseigné. Poids léger de 16.7 kg, volume moyen. Matériau so...

→ Categorie : Plastique
  Texte : Lot plastique à l'Usine A. Volume 64.7 L, poids 47.3 kg. Aspect indéterminée, rigidité moyenne. Aucu...

→ Categorie : Plastique
  Texte : Déchet plastique collecté à l'Usine B. Poids 33.0 kg, volume 44.0 L. Rigidité semi-rigide, non condu...


In [4]:
def nettoyer_texte(texte):
    # 1. Minuscules
    texte = texte.lower()
    # 2. Supprimer chiffres et ponctuation
    texte = re.sub(r'[^a-zàâäéèêëîïôùûüç\s]', ' ', texte)
    # 3. Supprimer espaces multiples
    texte = re.sub(r'\s+', ' ', texte).strip()
    return texte

# Tester
exemple = df_nlp['Rapport_Collecte'].iloc[0]
print("AVANT :", exemple[:100])
print("APRES :", nettoyer_texte(exemple)[:100])

AVANT : Lot de papier récupéré dans un site non renseigné. Poids léger de 16.7 kg, volume moyen. Matériau so
APRES : lot de papier récupéré dans un site non renseigné poids léger de kg volume moyen matériau souple non


In [5]:
# Stopwords français + domaine
stop_fr = set(stopwords.words('french'))
stop_domaine = {'lot', 'déchet', 'collecté', 'volume', 'poids', 
                'kg', 'litre', 'usine', 'site', 'matériau', 'aspect'}
tous_stopwords = stop_fr.union(stop_domaine)

# Stemmer français
stemmer = SnowballStemmer('french')

def pretraiter_texte(texte):
    # 1. Nettoyage
    texte = nettoyer_texte(texte)
    # 2. Tokenisation
    tokens = word_tokenize(texte, language='french')
    # 3. Suppression stopwords
    tokens = [t for t in tokens if t not in tous_stopwords]
    # 4. Stemming
    tokens = [stemmer.stem(t) for t in tokens]
    return ' '.join(tokens)

# Appliquer sur tout le dataset
df_nlp['texte_propre'] = df_nlp['Rapport_Collecte'].apply(pretraiter_texte)

print("✅ Prétraitement terminé !")
print("\nAvant :", df_nlp['Rapport_Collecte'].iloc[0][:80])
print("Après :", df_nlp['texte_propre'].iloc[0][:80])

✅ Prétraitement terminé !

Avant : Lot de papier récupéré dans un site non renseigné. Poids léger de 16.7 kg, volum
Après : papi récuper non renseign leg moyen soupl non conducteur tres opaqu bon état gén


In [6]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split

# Cible
y = df_nlp['Categorie']
X_text = df_nlp['texte_propre']

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_text, y, test_size=0.20, random_state=42, stratify=y
)

# Bag of Words
bow = CountVectorizer()
X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

# TF-IDF
tfidf = TfidfVectorizer(ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"✅ Vectorisation terminée")
print(f"BOW shape   : {X_train_bow.shape}")
print(f"TF-IDF shape: {X_train_tfidf.shape}")

✅ Vectorisation terminée
BOW shape   : (5432, 114)
TF-IDF shape: (5432, 646)


In [7]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

classificateurs = {
    'Naive Bayes'        : MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'LinearSVC'          : LinearSVC(),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=42)
}

print("=== BOW ===")
for nom, clf in classificateurs.items():
    clf.fit(X_train_bow, y_train)
    y_pred = clf.predict(X_test_bow)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    print(f"{nom:25s} → Accuracy: {acc:.4f} | F1: {f1:.4f}")

print("\n=== TF-IDF ===")
classificateurs2 = {
    'Naive Bayes'        : MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'LinearSVC'          : LinearSVC(),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=42)
}
for nom, clf in classificateurs2.items():
    clf.fit(X_train_tfidf, y_train)
    y_pred = clf.predict(X_test_tfidf)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    print(f"{nom:25s} → Accuracy: {acc:.4f} | F1: {f1:.4f}")

=== BOW ===
Naive Bayes               → Accuracy: 1.0000 | F1: 1.0000
Logistic Regression       → Accuracy: 1.0000 | F1: 1.0000
LinearSVC                 → Accuracy: 1.0000 | F1: 1.0000
Random Forest             → Accuracy: 1.0000 | F1: 1.0000

=== TF-IDF ===
Naive Bayes               → Accuracy: 1.0000 | F1: 1.0000
Logistic Regression       → Accuracy: 1.0000 | F1: 1.0000
LinearSVC                 → Accuracy: 1.0000 | F1: 1.0000
Random Forest             → Accuracy: 1.0000 | F1: 1.0000


In [9]:
# Word2Vec non disponible pour Python 3.14
# On utilise TF-IDF avec SVD comme alternative (LSA)
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline

# LSA = TF-IDF + réduction dimensionnelle (équivalent sémantique)
lsa_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2))),
    ('svd', TruncatedSVD(n_components=100, random_state=42))
])

X_train_lsa = lsa_pipeline.fit_transform(X_train)
X_test_lsa = lsa_pipeline.transform(X_test)

print(f"✅ LSA (alternative Word2Vec) shape: {X_train_lsa.shape}")

✅ LSA (alternative Word2Vec) shape: (5432, 100)


In [10]:
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

print("=== LSA (alternative Word2Vec) ===")
for nom, clf in {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'LinearSVC': LinearSVC()
}.items():
    clf.fit(X_train_lsa, y_train)
    y_pred = clf.predict(X_test_lsa)
    acc = accuracy_score(y_test, y_pred)
    print(f"{nom:25s} → Accuracy: {acc:.4f}")

=== LSA (alternative Word2Vec) ===
Logistic Regression       → Accuracy: 1.0000
LinearSVC                 → Accuracy: 1.0000


In [11]:
# FastText simulé avec caractère n-grammes
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

# TF-IDF avec n-grammes de CARACTÈRES (comme FastText)
fasttext_like = TfidfVectorizer(
    analyzer='char_wb',    # n-grammes de caractères
    ngram_range=(3, 5),    # trigrammes à 5-grammes
    min_df=2
)

X_train_ft = fasttext_like.fit_transform(X_train)
X_test_ft = fasttext_like.transform(X_test)

clf_ft = LinearSVC()
clf_ft.fit(X_train_ft, y_train)
y_pred_ft = clf_ft.predict(X_test_ft)

acc = accuracy_score(y_test, y_pred_ft)
print(f"✅ FastText-like (char n-grammes) → Accuracy: {acc:.4f}")
print(f"   Avantage : robuste aux fautes d'orthographe")

✅ FastText-like (char n-grammes) → Accuracy: 1.0000
   Avantage : robuste aux fautes d'orthographe


In [12]:
from sentence_transformers import SentenceTransformer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

print("⏳ Chargement CamemBERT (première fois = téléchargement ~500MB)...")

# Charger le modèle CamemBERT
model_bert = SentenceTransformer('camembert-base')

print("✅ CamemBERT chargé !")

# Vectoriser les textes
print("⏳ Vectorisation en cours...")
X_train_bert = model_bert.encode(X_train.tolist(), show_progress_bar=True)
X_test_bert = model_bert.encode(X_test.tolist(), show_progress_bar=True)

print(f"✅ BERT shape: {X_train_bert.shape}")

⏳ Chargement CamemBERT (première fois = téléchargement ~500MB)...


config.json:   0%|          | 0.00/508 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] CamembertModel LOAD REPORT from: camembert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

tokenizer.json: 0.00B [00:00, ?B/s]

✅ CamemBERT chargé !
⏳ Vectorisation en cours...


Batches:   0%|          | 0/170 [00:00<?, ?it/s]

Batches:   0%|          | 0/43 [00:00<?, ?it/s]

✅ BERT shape: (5432, 768)
